# TabDPT Classifier Artifact Inference — DIMER tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tabdpt-classifier-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tabdpt-classifier-pipeline/blob/main/tutorials/tabdpt_classifier_artifact_inference_colab.ipynb)

**Profile:** `ARTIFACT-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification `1.0`  
**Repository code revision exercised:** `3f40bb364c2cc72d5e366910172bff92e1d8d5a2`

This notebook consumes an **externally supplied** `tabdpt-dimer-context-v3` artifact (`artifact.json` plus `training_context.parquet`) from a separate producing execution. It validates the complete production runtime/preprocessing/model contract before reconstruction, restores the fitted preprocessing and support context, and scores **genuinely new input** without refitting preprocessing or gradient-training model weights.

**By the end of this notebook you will be able to:** validate an external DIMER v3 artifact; inspect model identity, format, complete runtime controls, schema and context integrity; reconstruct serving state; validate and score external unlabelled CSV data; and export predictions plus provenance.

This notebook does not create its own artifact and does not demonstrate model selection, calibrated probabilities, or production fitness. References: repository README, MODEL_CARD.md, TABULAR_CLASSIFICATION_DATASET_SPEC.md, DIMER_CONTRACT.md, the pinned TabDPT upstream source, and `Layer6/TabDPT`.


## Prerequisites and trust boundary

Use a fresh Colab or compatible Jupyter runtime with Python 3.11–3.13. GPU is recommended; CPU is supported but slower. `use_flash=False` is the portable demonstrated path. Supply exactly `artifact.json`, `training_context.parquet`, and later one new unlabelled CSV. Uploaded files stay in the notebook runtime unless explicitly exported.

The pinned code SHA is retained by branch `anchors/notebook-spec-v1-code-20260910`; that branch preserves reachability but the immutable commit remains the version identifier.

**Trust boundary:** JSON/Parquet structure, size, path, model-identity, and SHA-256 checks establish internal consistency, not sender authenticity. The artifact is data-only, but its support table may contain governed source data. The separately acquired base checkpoint is immutable and digest-verified. This notebook intentionally accepts individual files rather than ZIP/TAR archives.

Runtime controls are read from the artifact's complete production `runtimeConfig`: target/drop settings, support/validation controls, fine-tuning flag, ensemble/context/batch settings, temperature, and seed. Seeded execution does not promise bitwise equality across devices or library/kernel builds.


In [ ]:
import sys
if "torch" in sys.modules:
    raise RuntimeError("Start from a fresh runtime: install pinned dependencies before importing PyTorch.")
REPO_REVISION = "3f40bb364c2cc72d5e366910172bff92e1d8d5a2"
REPO_DIR = "/content/tabdpt-classifier-pipeline"
!rm -rf "$REPO_DIR"
!git clone -q https://github.com/kurtvalcorza/tabdpt-classifier-pipeline.git "$REPO_DIR"
!git -C "$REPO_DIR" checkout -q "$REPO_REVISION"
!python -m pip install -q -r "$REPO_DIR/tutorials/requirements-colab.txt"
!python -m pip install -q --no-deps "$REPO_DIR"


## 1. Verify runtime and expected model identity

Report the effective environment and verify the immutable TabDPT checkpoint before accepting an artifact. Successful checksum verification proves model-byte identity, not model quality.


In [ ]:
import importlib.metadata as mdlib
import platform
import torch
from tabdpt_classifier_pipeline import (
    TABDPT_HF_REPO, TABDPT_HF_REVISION, TABDPT_UPSTREAM_CODE_COMMIT,
    TABDPT_WEIGHT_FILENAME, TABDPT_WEIGHT_SHA256, TabDPTClassificationPipeline,
    resolve_tabdpt_weights, validate_dimer_artifact,
)
print("Python:", sys.version.split()[0]); print("Platform:", platform.platform())
for package in ["tabdpt", "torch", "numpy", "pandas", "scikit-learn", "huggingface-hub", "pyarrow"]:
    print(f"{package}:", mdlib.version(package))
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"); print("CUDA:", torch.version.cuda)
print("Execution assumptions: compile_model=False, use_flash=False, quantization=None, precision=framework/device default")
weights = resolve_tabdpt_weights()
print("Expected model:", TABDPT_HF_REPO, TABDPT_HF_REVISION, TABDPT_WEIGHT_FILENAME, TABDPT_WEIGHT_SHA256, TABDPT_UPSTREAM_CODE_COMMIT)
print("Verified checkpoint path:", weights)


## 2. Supply and validate the external artifact

Upload `artifact.json` and `training_context.parquet` from a separate E2E or DIMER producer. The shared validator runs before model reconstruction and rejects format/task/model mismatches, reduced or inconsistent runtime metadata, fitted-preprocessing inconsistencies, unsafe paths/symlinks, oversized files, size/digest mismatches, and unexpected files.


In [ ]:
from pathlib import Path
import shutil
from google.colab import files
ARTIFACT_DIR = Path("/content/external-tabdpt-artifact")
shutil.rmtree(ARTIFACT_DIR, ignore_errors=True); ARTIFACT_DIR.mkdir(parents=True)
uploaded = files.upload()
expected = {"artifact.json", "training_context.parquet"}; received = {Path(name).name for name in uploaded}
if received != expected:
    raise ValueError(f"Upload exactly {sorted(expected)}; received {sorted(received)}")
for name, payload in uploaded.items():
    (ARTIFACT_DIR / Path(name).name).write_bytes(payload)
manifest_path = ARTIFACT_DIR / "artifact.json"
manifest, context_path = validate_dimer_artifact(manifest_path, strict_directory=True)
runtime_config = manifest["runtimeConfig"]; preprocessing = manifest["preprocessing"]
print("Artifact format/task:", manifest["format"], manifest["taskType"])
print("Base model/revision:", manifest["baseModel"]["repo"], manifest["baseModel"]["revision"])
print("Weight SHA-256:", manifest["baseModel"]["sha256"])
print("Target/drop columns:", runtime_config["target_column"], runtime_config["drop_columns"])
print("Support/validation controls:", runtime_config["max_train_rows"], runtime_config["validation_split"])
print("Inference controls:", {k: runtime_config[k] for k in ("n_ensembles", "context_size", "batch_size", "temperature", "seed")})
print("Context bytes/SHA-256:", context_path.stat().st_size, manifest["trainingContext"]["sha256"])
print("Classes/features:", manifest["classNames"], preprocessing["encoder"]["featureColumns"])


## 3. Reconstruct the serving state

`load_artifact()` restores fitted feature encoding and support context from the validated files. It does not refit preprocessing from inference data. The upstream estimator's support registration is in-context conditioning, not gradient training.


In [ ]:
pipe = TabDPTClassificationPipeline.load_artifact(manifest_path, compile_model=False, use_flash=False, seed=runtime_config["seed"])
print("Restored target/classes/features:", pipe.target_column, pipe.class_labels_, pipe.feature_encoder.feature_columns)


## 4. Upload and validate genuinely new input

Upload one external unlabelled CSV that was not used to produce the artifact. Duplicate headers are rejected before pandas can rename them; the target must be absent; effective feature names must exactly match fitted preprocessing. Unknown categorical values use the restored unknown-category code and are reported as drift.


In [ ]:
import csv
from collections import Counter
import pandas as pd
INPUT_DIR = Path("/content/tabdpt-inference-input"); shutil.rmtree(INPUT_DIR, ignore_errors=True); INPUT_DIR.mkdir(parents=True)
uploaded_input = files.upload()
csv_names = [Path(name).name for name in uploaded_input if Path(name).suffix.lower() == ".csv"]
if len(uploaded_input) != 1 or len(csv_names) != 1:
    raise ValueError("Upload exactly one new unlabelled CSV")
input_path = INPUT_DIR / csv_names[0]; input_path.write_bytes(next(iter(uploaded_input.values())))
with input_path.open("r", encoding="utf-8-sig", newline="") as handle:
    header = next(csv.reader(handle), None)
if not header:
    raise ValueError("Inference CSV is empty")
duplicates = sorted(k for k, v in Counter(header).items() if v > 1)
if duplicates:
    raise ValueError(f"Duplicate column names: {duplicates}")
new_data = pd.read_csv(input_path)
if pipe.target_column in new_data.columns:
    raise ValueError(f"Remove target column {pipe.target_column!r} from inference input")
effective = new_data.drop(columns=pipe.drop_columns_, errors="ignore"); required = list(pipe.feature_encoder.feature_columns)
missing = [c for c in required if c not in effective.columns]; extra = [c for c in effective.columns if c not in required]
if missing or extra:
    raise ValueError(f"Feature schema mismatch; missing={missing}, extra={extra}")
for col, mapping in pipe.feature_encoder.category_maps.items():
    unseen = sorted({str(v) for v in effective[col].dropna().tolist()} - set(mapping))
    if unseen:
        print(f"WARNING: {col!r} unseen categorical values: {unseen[:10]}")
print("Validated new input shape:", new_data.shape)


## 5. Predict and export machine-readable results

Inference parameters are derived only from the validated artifact runtime contract. The discrete rule is argmax over class scores. Scores are probability-normalized but this tutorial does not establish calibration or interpret them as guaranteed confidence estimates.


In [ ]:
import hashlib, json
inference_kwargs = {key: runtime_config[key] for key in ("n_ensembles", "context_size", "batch_size", "temperature", "seed")}
if len(pd.read_parquet(context_path)) > runtime_config["context_size"]:
    print("Context reduction active: seeded balanced support subsampling occurs per ensemble according to artifact context_size.")
pred = pipe.predict(new_data, **inference_kwargs); scores = pipe.predict_proba(new_data, **inference_kwargs)
result = pd.DataFrame({"row_id": new_data.index.astype(str), "prediction": pred.astype(str)})
for label in pipe.class_labels_:
    result[f"score_{label}"] = scores[label].to_numpy()
OUTPUT_DIR = Path("/content/tabdpt-artifact-inference-output"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pred_path = OUTPUT_DIR / "predictions.csv"; prov_path = OUTPUT_DIR / "provenance.json"; result.to_csv(pred_path, index=False)
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()
provenance = {
    "notebookProfile": "ARTIFACT-INFERENCE", "notebookSpec": "1.0", "repositoryRevision": REPO_REVISION,
    "artifact": {"format": manifest["format"], "manifestSha256": sha256_file(manifest_path), "contextSha256": manifest["trainingContext"]["sha256"]},
    "model": manifest["baseModel"], "runtimeConfig": runtime_config, "classOrder": list(pipe.class_labels_),
    "runtime": {"python": sys.version.split()[0], "platform": platform.platform(), "torch": mdlib.version("torch"), "tabdpt": mdlib.version("tabdpt"), "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU", "cuda": torch.version.cuda, "use_flash": False, "compile_model": False, "quantization": None, "precision": "framework/device default"},
    "input": {"filename": input_path.name, "sha256": sha256_file(input_path), "rows": len(new_data), "columns": list(new_data.columns)},
    "decisionRule": "argmax over class-score columns", "calibrationEstablished": False,
}
prov_path.write_text(json.dumps(provenance, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(result.head()); print("Predictions:", pred_path); print("Provenance:", prov_path)


## Interpretation and troubleshooting

A successful run demonstrates that this pinned code path can validate a separately produced production-shaped DIMER v3 artifact, reconstruct serving state from artifact contents plus the explicitly permitted pinned base model, validate genuinely new input, and export predictions/provenance. It does **not** prove accuracy, calibration, fairness, robustness, sender authenticity, or production fitness. No evaluation metric is reported because the required input is unlabelled.

Provenance/digest/size/runtime-contract failures mean the artifact must not be reconstructed; schema failures mean the new input does not match fitted preprocessing. Memory pressure should be addressed by a newly produced artifact or governed deployment configuration rather than silently changing recorded serving semantics. Clean-runtime execution for the exact release revision remains the final release gate.
